## 1. Importación y Unificación de Datos

En esta primera sección importamos las librerías necesarias, localizamos todos los archivos CSV procedentes del web scraping y los unificamos en un único DataFrame. Además, aplicamos una primera capa de limpieza eliminando filas duplicadas y borrando información sensible (como los nombres de usuario) para cumplir con el RGPD, guardando un archivo en bruto (raw) combinado.

In [ ]:
# Librerías
import pandas as pd
import glob
import os
import re
import emoji

# 1. Le decimos a Python dónde están los archivos 
ruta = "datos/scraping_hoteles/*.csv" 

# 2. Buscamos todos los archivos CSV de la carpeta
archivos_csv = glob.glob(ruta)
print(f"He encontrado {len(archivos_csv)} archivos CSV para unir.")

# 3. Creamos una lista vacía y vamos leyendo y guardando cada CSV
lista_dataframes = [pd.read_csv(archivo) for archivo in archivos_csv]

# 4. Unimos todos los recortes en una sola tabla gigante
df_total = pd.concat(lista_dataframes, ignore_index=True)
print(f"Total de reseñas combinadas en bruto: {len(df_total)}")

# 5. Eliminar duplicados (mismo hotel o reseña 2 veces)
df_total = df_total.drop_duplicates()

# 6. Ética y Privacidad (Borramos el nombre del usuario si existe)
nombre_columna_usuario = 'nombre_usuario' 
if nombre_columna_usuario in df_total.columns:
    df_total = df_total.drop(columns=[nombre_columna_usuario])
    print("✓ Nombres de usuario borrados por privacidad (Cumplimiento RGPD).")

print(f"Total de reseñas finales listas para NLP: {len(df_total)}")

# 7. Exportamos el súper archivo final
df_total.to_csv("datos/procesados/df_hoteles_vlc_completo.csv", index=False, encoding='utf-8-sig')
print("¡Archivo 'df_hoteles_vlc_completo.csv' creado con éxito!")

# Hacemos una copia del dataframe original para no perderlo
df_original = df_total.copy()

## 2. Limpieza Estructural Básica

Aquí eliminamos las columnas que no aportan valor analítico (como el orden del scraper) y normalizamos valores anómalos o irrelevantes en variables categóricas (como extraer solo el número en la columna de 'noches', quitar prefijos en 'personas' o tratar los puntos solitarios como nulos en los comentarios).

In [ ]:
# Eliminar columna de web order
if 'web_scraper_order' in df_total.columns:
    df_total = df_total.drop(columns=['web_scraper_order'])

# Extraer el número de noches y pasar el dígito a entero
df_total['noches'] = df_total['noches'].str.extract(r'(\d+)').astype(int)

# Quitar el "En " de la columna de personas para dejar solo la categoría
df_total['personas'] = df_total['personas'].str.replace('En ', '')

# Si la columna de negativo o positivo SOLO tiene un punto, lo convertimos a NA (nulo)
df_total['negativo'] = df_total['negativo'].replace('.', pd.NA)
df_total['positivo'] = df_total['positivo'].replace('.', pd.NA)

display(df_total.head(3))

## 3. Transformación de Variables Numéricas y Fechas

En este apartado estandarizamos los tipos de datos. Convertimos las valoraciones (notas) de formato texto europeo (con comas) a numérico (con puntos) y transformamos la columna de fechas, limpiando el texto adicional y pasando los meses al formato adecuado para convertirlo en un objeto datetime de Pandas.

In [ ]:
# Convertir columnas de notas a numéricas
columnas_notas = [col for col in df_total.columns if col.startswith('nota')]
for columna in columnas_notas:
    df_total[columna] = df_total[columna].str.replace(',', '.')
    df_total[columna] = pd.to_numeric(df_total[columna], errors='coerce')

# Procesar la columna de fechas
# Eliminar parte "Fecha del comentario: " y espacios
df_total['fecha'] = df_total['fecha'].apply(lambda x: re.sub('Fecha del comentario: ', '', str(x)).strip())

# Mapear los meses en español a código numérico
mes_a_codigo = {
    'enero': '01', 'febrero': '02', 'marzo': '03', 'abril': '04', 
    'mayo': '05', 'junio': '06', 'julio': '07', 'agosto': '08',
    'septiembre': '09', 'octubre': '10', 'noviembre': '11', 'diciembre': '12'
}

for mes, codigo in mes_a_codigo.items():
    df_total['fecha'] = df_total['fecha'].apply(lambda x: re.sub(mes, codigo, str(x)))

# Pasar a formato fecha (datetime)
df_total['fecha'] = pd.to_datetime(df_total['fecha'], format='%d de %m de %Y', errors='coerce')

display(df_total[['fecha'] + columnas_notas].head(3))

## 4. División Relacional del Dataset (Hoteles y Comentarios)

Para evitar redundancia de datos y facilitar análisis posteriores, dividimos el DataFrame unificado en dos tablas relacionadas mediante el nombre del hotel: 
- una con la información estática del alojamiento y sus notas medias
- otra con el desglose de cada comentario individual.

In [ ]:
# Creamos el csv de hoteles (solo información agregada)
columnas_hoteles = ['web_scraper_start_url', 'nombre_del_hotel'] + [col for col in df_total.columns if col.startswith('nota_')]
df_hoteles = df_total[columnas_hoteles].drop_duplicates()
df_hoteles.to_csv("datos/procesados/df_hoteles_vlc_info.csv", index=False, encoding='utf-8-sig')
print("¡Archivo 'df_hoteles_vlc_info.csv' creado con éxito!")

# Creamos el csv de comentarios (dejamos 'nombre_del_hotel' como clave foránea)
columnas_comentarios = ['nombre_del_hotel'] + [col for col in df_total.columns if col not in columnas_hoteles]
df_comentarios = df_total[columnas_comentarios].copy()
df_comentarios.to_csv("datos/procesados/df_hoteles_vlc_comentarios.csv", index=False, encoding='utf-8-sig')
print("¡Archivo 'df_hoteles_vlc_comentarios.csv' creado con éxito!")